# RAG Coding Assessment Prep (Python, from scratch)

**Goal:** sit down next week and independently implement ingestion, preprocessing, chunking, FastEmbed embeddings, an in-memory vector index, cosine search, top-k retrieval, and modular Python -- without LangChain, LlamaIndex, or AI assistance.

This is **not** a Python textbook and **not** a RAG theory course. It is an assessment workbook: syntax, patterns, edge cases, hidden tests, and the exact primitives interviewers ask you to type.

You already know DSA from C++. Every Python section maps the new syntax to that mental model, then immediately uses it on RAG data.

## How to use this notebook

1. Read a concept, run the cells, then **close the solution** and re-implement from memory.
2. Treat red assertions as the contract. Assessments hide extra cases; this notebook lists them.
3. FastEmbed cells need `pip install fastembed numpy`. Everything else runs on the standard library + NumPy.
4. A companion package lives in `rag/` -- that is the Part 14 system. Learn it here first; do not import it during a mock timed drill.
5. Official FastEmbed API was verified against [qdrant/fastembed](https://github.com/qdrant/fastembed) (current default model `BAAI/bge-small-en-v1.5`).

## Table of contents

| Part | Topic |
|------|--------|
| 1 | Python building blocks for RAG coding |
| 2 | How Python represents RAG data |
| 3 | RAG-relevant DSA (hashing, windows, top-k) |
| 4 | NumPy + cosine similarity |
| 5 | FastEmbed (current API, not magic) |
| 6 | Ingestion + preprocessing |
| 7 | Chunking |
| 8 | Embedding pipeline |
| 9 | In-memory vector index |
| 10 | Retrieval and ranking |
| 11 | Module separation |
| 12 | Experienced-engineer mistakes |
| 13 | Testing / hidden cases |
| 14 | Mini-RAG from primitives |
| 15 | Timed assessment simulations (questions first, solutions after) |

## Recurring cell template

For every important idea this notebook uses:

1. What is it?
2. Why do I need it?
3. Syntax
4. Simple example
5. RAG-related example
6. Common mistakes
7. Edge cases
8. Hidden-test cases
9. Complexity
10. Practice problem
11. Assessment takeaway


In [ ]:
from __future__ import annotations

import json
import math
import heapq
import re
from collections import Counter, defaultdict
from pathlib import Path
from dataclasses import dataclass, field
from typing import Any, Iterable

import numpy as np

print("numpy", np.__version__)


# Part 1 — Python building blocks

C++ → Python cheat sheet you will use constantly:

| C++ | Python |
|-----|--------|
| `int x = 3;` | `x = 3` (runtime type, name is a reference) |
| `std::vector<T>` | `list` (mutable, dynamic, heterogeneous) |
| `std::array` / `std::pair` | `tuple` (immutable) |
| `std::unordered_map` | `dict` (insertion-ordered since 3.7) |
| `std::unordered_set` | `set` |
| `std::string` | `str` (immutable Unicode) |
| `nullptr` | `None` |
| RAII / scope guard | `with` |
| `const T&` default copy of containers | **assignment copies the reference**, not the object |
| `vector.at(i)` | `lst[i]` (IndexError, no `at`) |
| `map.count(k)` / `map[k]` inserts | `k in d`, `d.get(k)`, `d[k]` raises KeyError |

**Mutable vs immutable (this fails people in assessments):**

- Immutable: `int`, `float`, `bool`, `str`, `tuple`, `frozenset`
- Mutable: `list`, `dict`, `set`, custom objects, NumPy arrays
- Binding `b = a` does **not** copy a list/dict. Use `a.copy()`, `list(a)`, or `copy.deepcopy` for nested structures.


## 1.1 Variables and data types

**What:** a name bound to an object. The object has a type; the name does not.

**Why:** RAG records are nested dicts/lists. If you mutate a shared metadata dict, every chunk silently changes.


In [ ]:
# Syntax / simple examples
n = 7                  # int (unlimited size, unlike 32/64-bit C++)
pi = 3.14              # float (IEEE-754 double)
ok = True              # bool, subclass of int
name = "doc1"          # str
nothing = None         # NoneType

print(type(n), type(name), type(nothing))
print(True + True)     # 2  — bool is an int. Do not use this in RAG code.

# Identity vs equality
a = [1, 2]
b = a
c = [1, 2]
print(a == c, a is c, a is b)  # True False True


In [ ]:
# RAG-related: shared mutable metadata is a landmine
shared_meta = {"lang": "en"}
doc_a = {"id": "a", "metadata": shared_meta}
doc_b = {"id": "b", "metadata": shared_meta}
doc_a["metadata"]["lang"] = "fr"
print(doc_b["metadata"])  # {'lang': 'fr'}  — both changed

# Pythonic: copy at the boundary
doc_b = {"id": "b", "metadata": dict(shared_meta)}


**Common mistakes:** using `is` for equality (`x is 1000` can be False); treating `bool` as not an int; forgetting `None` is not `0` or `""`.

**Edge / hidden:** `bool("False")` is `True` (non-empty string). `int(True) == 1`. Huge ints never overflow.

**Complexity:** name lookup O(1) average.

**Practice:** bind two documents to the same list of tags, append a tag to one, predict the other.

**Takeaway:** copy mutable fields when you attach them to a new document/chunk.


## 1.2 Function parameters

**What:** Python arguments are passed by **object reference**. Rebinding a parameter does not rebind the caller's name; mutating a mutable object does.

| Style | C++ analogue | Example |
|-------|----------------|---------|
| positional | normal args | `f(1, 2)` |
| keyword | designated-ish | `f(b=2, a=1)` |
| default | default args | `def f(a, b=3)` |
| `*args` | variadic | extra positionals as tuple |
| `**kwargs` | not really | extra keywords as dict |

**Syntax**
```python
def load(path, encoding="utf-8", *, recursive=False, **kwargs):
    ...
```
The `*` makes `recursive` keyword-only (good API hygiene).


In [ ]:
def greet(name, greeting="hello", *extras, shout=False, **meta):
    text = f"{greeting}, {name}"
    if shout:
        text = text.upper()
    return text, extras, meta

print(greet("Ada"))
print(greet("Ada", "hi", "note", shout=True, source="file.txt"))

# Default-argument trap (the #1 Python interview miss)
def add_chunk(chunk, bucket=[]):  # BAD: one list shared across calls
    bucket.append(chunk)
    return bucket

print(add_chunk("a"))
print(add_chunk("b"))  # ['a', 'b']  — leaked state


def add_chunk_ok(chunk, bucket=None):
    if bucket is None:
        bucket = []
    bucket.append(chunk)
    return bucket


**RAG example:** `chunk_text(text, chunk_size=100, overlap=20)` — keyword args at the call site document intent.

**Mistakes:** mutable defaults; putting `*args` before you need it; `def f(a=1, b)` is a SyntaxError.

**Hidden:** `def f(**kwargs)` plus later `kwargs["top_k"]` without `.get` blows up on missing keys.

**Takeaway:** keyword-only flags after `*` (`*, top_k=5`). Never default a list/dict/set to a literal.


## 1.3 Lists

**What:** ordered, mutable sequence. C++ `vector`, but can hold mixed types.

**Why:** documents, chunks, ranked results — almost every RAG pipeline is a list of dicts.


In [ ]:
chunks = ["alpha", "beta", "gamma", "delta"]
print(chunks[0], chunks[-1])          # first / last
print(chunks[1:3])                    # [start:stop)  -> ['beta', 'gamma']
print(chunks[:2], chunks[2:], chunks[::2])
print(chunks[::-1])                   # reverse copy

chunks.append("eps")                  # O(1) amortized
chunks.extend(["z1", "z2"])           # concat in place
chunks.insert(0, "head")              # O(n)
item = chunks.pop()                   # last, O(1)
del chunks[0]                         # O(n)

# slicing copies
window = chunks[0:2]
window[0] = "MUTATED"
print(chunks[0])                      # original unchanged

# BUT elements are references
nested = [{"id": 1}, {"id": 2}]
alias = nested[0:1]
alias[0]["id"] = 99
print(nested[0])                      # {'id': 99}


In [ ]:
# BAD: mutate while iterating
ids = ["d1", "d2", "d3"]
for i, item in enumerate(ids):
    if item == "d2":
        ids.pop(i)   # skips elements — do not do this

# GOOD
ids = [x for x in ["d1", "d2", "d3"] if x != "d2"]
print(ids)


**Edge:** `lst[10:20]` on a short list returns `[]` (no error). `lst[10]` raises. `lst[-0]` is `lst[0]`.

**Hidden:** `lst += other` mutates in place (so it works on a list passed into a function); `lst = lst + other` rebinds.

**Complexity:** index O(1), append O(1) amortized, insert/pop-front O(n), `x in lst` O(n).

**Practice:** given tokens, return overlapping windows of size 3 step 2 using only slicing.

**Takeaway:** slicing is how you implement character chunking. Membership tests on long lists belong in a `set`.


## 1.4 Tuples

**What:** immutable sequence. C++ `tuple` / `pair`.

**Why:** heap entries `(score, chunk_id, index)`, function multi-returns, dict keys.


In [ ]:
record = ("doc1_chunk_0", 0.87)
chunk_id, score = record              # unpacking
print(record[0])

# Single-element tuple needs a comma
not_a_tuple = ("only")
is_a_tuple = ("only",)
print(type(not_a_tuple), type(is_a_tuple))

# Tuple is immutable, but it can CONTAIN mutables
pair = ("doc", {"lang": "en"})
pair[1]["lang"] = "de"                # allowed
# pair[0] = "other"                   # TypeError

# Dict key must be hashable — tuple of immutables works
seen = {("doc1", 0): True}


**Mistakes:** `("x")` is a string; mutating an object *inside* a tuple and thinking the tuple is a deep snapshot.

**Takeaway:** return `(chunk, score)` tuples from search, or a small dataclass. Prefer dataclasses once fields grow.


## 1.5 Sets

**What:** unordered unique hashables. C++ `unordered_set`.

**Why:** duplicate document IDs, visited characters in a sliding window, stopword filters, O(1) membership.


In [ ]:
ids = ["a", "b", "a", "c"]
print(set(ids))                       # uniqueness
print(len(ids) != len(set(ids)))      # True => contains duplicate

allowed = {".txt", ".json"}
print(".md" in allowed)               # O(1) avg

# set operations
a, b = {"doc1", "doc2"}, {"doc2", "doc3"}
print(a | b, a & b, a - b)

# unhashable elements fail
try:
    {{"id": 1}}                       # TypeError
except TypeError as e:
    print("expected:", e)

# Pythonic duplicate check
def has_duplicate_ids(docs):
    seen = set()
    for d in docs:
        if d["id"] in seen:
            return True
        seen.add(d["id"])
    return False


**Edge:** empty set is `set()`, not `{}` (`{}` is a dict). Sets are unordered — do not rely on print order in tests (3.7+ dicts are ordered; sets are not).

**Hidden:** `set("aba")` is `{"a", "b"}` because strings are iterable.

**Complexity:** add/in/remove O(1) average, O(n) worst.

**Takeaway:** the instant you write `if id in list_of_ids` inside a loop, you wanted a set.


## 1.6 Dictionaries

**What:** hash map. The RAG record type.

**Why:** documents, chunks, embeddings-by-id, frequency counts, inverted indexes.


In [ ]:
doc = {"id": "doc1", "text": "hello", "source": "a.txt"}
print(doc["id"])
print(doc.get("lang"))                # None, no exception
print(doc.get("lang", "en"))          # default
doc.setdefault("metadata", {})        # insert only if missing

# KeyError is the silent-test killer
try:
    _ = doc["missing"]
except KeyError as e:
    print("KeyError", e)

# iteration
for key, value in doc.items():
    pass
print(list(doc.keys()), list(doc.values()))

# merge (3.9+)
defaults = {"lang": "en", "source": "unknown"}
print({**defaults, **doc})            # doc wins
print(defaults | doc)                 # same

# frequency counting — two styles
text = "to be or not to be"
freq = {}
for w in text.split():
    freq[w] = freq.get(w, 0) + 1
print(freq)
print(Counter(text.split()))


In [ ]:
# BAD: d[k] for optional metadata
chunk = {"id": "c0", "text": "hi"}
# lang = chunk["metadata"]["lang"]    # KeyError

# GOOD
lang = chunk.get("metadata", {}).get("lang", "en")
print(lang)

# mutating during .items() can raise RuntimeError
d = {"a": 1, "b": 2}
# for k in d:
#     d[k + k] = 0                   # RuntimeError


**Edge:** keys must be hashable. `dict.fromkeys(["a", "b"], [])` shares one list. Insertion order is preserved (use it for deterministic tie-breaks).

**Complexity:** get/set O(1) average.

**Practice:** invert `{chunk_id: doc_id}` to `{doc_id: [chunk_ids]}`.

**Takeaway:** `.get` / `.setdefault` / `Counter` / `defaultdict(list)` are the assessment primitives. Write them until they are muscle memory.


## 1.7 Strings

**What:** immutable Unicode sequence.

**Why:** every document is a string until it becomes a vector.


In [ ]:
s = "Retrieval-Augmented Generation"
print(s.lower(), s.upper())
print(s.startswith("Retr"), s.endswith("tion"))
print("Aug" in s)
print(s.split())                      # whitespace split
print("a,b,,c".split(","))            # ['a', 'b', '', 'c']
print("  hi\n".strip())
print(" ".join(["chunk", "0"]))
print(s[0:10])                        # character slice (Unicode code points)
print(len("naïve"))                   # 5 — len is code points, not bytes

# useful RAG cleanups
messy = "Hello,\n\n   world!\tworld"
print(re.sub(r"\s+", " ", messy).strip())

# immutable
t = s.replace("Generation", "Search")
print(s is t)


**Mistakes:** `for line in file` keeps `\n`; comparing unstripped IDs; `text.split("")` is illegal; lowercasing then expecting "US" and "us" to stay distinct.

**Hidden:** `"ß".upper()` is `"SS"`; `"İ".lower()` can be `"i̇"` (two chars). Assessments with Unicode exist.

**Takeaway:** normalize **whitespace** by default; do not lowercase or strip punctuation unless the prompt says so.


## 1.8 Loops

```python
for item in iterable: ...
for i, item in enumerate(iterable, start=0): ...
while cond: ...
```

`for` in Python is "range-for", not C `for(i=0;...)`. Use `range(n)` when you need indices, `enumerate` when you need both.


In [ ]:
docs = [{"id": "d1", "text": "aa"}, {"id": "d2", "text": "bb"}]

# Pythonic
for i, doc in enumerate(docs):
    print(i, doc["id"])

# C++-shaped (works, noisier)
for i in range(len(docs)):
    print(i, docs[i]["id"])

# zip — lockstep
ids = ["d1", "d2"]
texts = ["aa", "bb"]
for doc_id, text in zip(ids, texts):
    print(doc_id, text)

# zip stops at the SHORTEST — hidden test
print(list(zip([1, 2, 3], ["a"])))          # [(1, 'a')]
print(list(zip([1, 2, 3], ["a"], strict=True)))  # will raise in the next cell


In [ ]:
try:
    list(zip([1, 2, 3], ["a"], strict=True))
except ValueError as e:
    print("zip(strict=True):", e)


**Takeaway:** `enumerate` for chunk indexes; `zip(..., strict=True)` when aligning chunks to embeddings. A silent short zip is an alignment bug.


## 1.9 Comprehensions

```python
[expr for x in xs if cond]
{k: v for k, v in pairs}
{x for x in xs}
```

Prefer them when the body is one expression. Use a loop when you have side effects or need to break.


In [ ]:
texts = ["Hello", "", "World", "  "]
clean = [t.strip().lower() for t in texts if t.strip()]
print(clean)

# dict: id -> text
docs = [{"id": "d1", "text": "a"}, {"id": "d2", "text": "b"}]
by_id = {d["id"]: d["text"] for d in docs}
print(by_id)

# nested: flatten pages
pages = [["c0", "c1"], ["c2"]]
flat = [c for page in pages for c in page]
print(flat)

# BAD: side effects inside a comprehension
# [store.add(c) for c in chunks]      # do not


**Complexity:** still O(n). They are not "faster algorithms", they are faster to read.

**Takeaway:** filter empty chunks with a comprehension; never hide mutations in one.


## 1.10 File handling and 1.11 pathlib

C++ `ifstream` + RAII ≈ `with open(...)` / `Path.read_text()`.

Always use `pathlib.Path` in assessments. Always set `encoding="utf-8"` unless told otherwise.


In [ ]:
from tempfile import TemporaryDirectory

with TemporaryDirectory() as td:
    root = Path(td)
    p = root / "notes.txt"
    p.write_text("hello\nworld\n", encoding="utf-8")

    # GOOD
    with p.open("r", encoding="utf-8") as f:
        lines = f.read()
    print(repr(lines))
    print(p.read_text(encoding="utf-8").splitlines())

    # directory listing — deterministic
    (root / "b.txt").write_text("b", encoding="utf-8")
    (root / "a.txt").write_text("a", encoding="utf-8")
    print(sorted(root.glob("*.txt")))

    # BAD: open without encoding (locale-dependent on some machines)
    # BAD: f = open(p); data = f.read()   # leak if exception


**Edge:** missing file → `FileNotFoundError`. Is a directory → `IsADirectoryError`. Permission → `PermissionError`. Empty file → `""`.

**Hidden:** `Path.glob` is not sorted. Sort it. Binary files opened as text raise `UnicodeDecodeError`.

**Takeaway:** `Path`, `encoding="utf-8"`, `with`, `sorted(glob)`.


## 1.12 Exceptions

```python
try:
    ...
except FileNotFoundError as e:
    ...
except (UnicodeDecodeError, json.JSONDecodeError) as e:
    ...
else:
    ...   # no exception
finally:
    ...   # always
```

Assessments usually want **fail fast** on bad input, not "return None and continue". Swallowing errors hides missing documents.


In [ ]:
def parse_score(raw: str) -> float:
    try:
        value = float(raw)
    except (TypeError, ValueError) as e:
        raise ValueError(f"invalid score {raw!r}") from e
    if not math.isfinite(value):
        raise ValueError("score is NaN/Inf")
    return value

print(parse_score("0.87"))
try:
    parse_score("nan")
except ValueError as e:
    print(e)


**Takeaway:** catch the narrow type, wrap with context, do not `except Exception: pass` in a loader.


## 1.13 Modules and imports

```python
import json                         # module
from pathlib import Path            # name
from rag.preprocessing import chunker
from rag.preprocessing.chunker import chunk_text
```

- Package = directory with `__init__.py` (or namespace package).
- `from x import *` is banned in assessments.
- Circular imports: A imports B imports A — fix by pushing shared types to `models/`.

C++ `#include` copies text. Python `import` executes the module once and caches it in `sys.modules`.


## 1.14 JSON

| Call | Input | Output |
|------|--------|--------|
| `json.loads` | `str` | Python object |
| `json.load` | file object | Python object |
| `json.dumps` | object | `str` |
| `json.dump` | object + file | writes file |

JSON types: object→dict, array→list, string→str, number→int/float, true/false→bool, null→None.


In [ ]:
payload = {"id": "doc1", "text": "hello", "n": 3}
s = json.dumps(payload, ensure_ascii=False, indent=2)
print(s)
print(json.loads(s)["id"])

# keys become strings
print(json.loads(json.dumps({1: "a"})))  # {'1': 'a'}

# tuples become lists — hidden
print(json.loads(json.dumps(("a", "b"))))

try:
    json.loads("{bad")
except json.JSONDecodeError as e:
    print("malformed", e.msg)


**Takeaway:** `load`/`dump` for files, `loads`/`dumps` for strings. The extra `s` means "string".


## 1.15–1.18 Iterators, enumerate, zip, sorted

An **iterator** is lazy. FastEmbed `embed()` returns a generator: nothing is computed until you iterate. Forgetting `list(...)` is the #1 FastEmbed bug.

`sorted(xs, key=fn, reverse=False)` returns a **new list**. `list.sort` is in-place.

`sorted` is stable.


In [ ]:
def fake_embed(texts):
    print("computing")
    for t in texts:
        yield [float(len(t))]

gen = fake_embed(["a", "bb"])
print("created, not computed yet")
print(list(gen))
print(list(gen))   # []  — generators exhaust

results = [
    {"id": "b", "score": 0.9},
    {"id": "a", "score": 0.9},
    {"id": "c", "score": 0.1},
]
# rank by score desc, then id asc (deterministic ties)
ranked = sorted(results, key=lambda r: (-r["score"], r["id"]))
print(ranked)

print(list(enumerate(["chunk"], start=1)))
print(list(zip(["c0", "c1"], [[0.1], [0.2]])))


## 1.19–1.20 Lambda and custom sorting

`lambda args: expr` is an anonymous function. Use it for `key=` only. Named `def` otherwise.

`key` is called **once per element** (Schwartzian). Do not sort by a comparison callback like C++ `sort(cmp)` — Python 3 removed that; use `key` or `functools.cmp_to_key`.


In [ ]:
chunks = [
    {"id": "doc1_chunk_10", "score": 0.4, "index": 10},
    {"id": "doc1_chunk_2", "score": 0.4, "index": 2},
]
print(sorted(chunks, key=lambda c: (-c["score"], c["index"])))

# BAD: sorting the formatted id as a string
print(sorted(["chunk_2", "chunk_10"]))   # ['chunk_10', 'chunk_2']


## 1.21 NumPy basics (preview — full treatment in Part 4)

`np.array` is a typed, contiguous block. RAG vectors are `shape (d,)` or a batch `(n, d)`.


In [ ]:
v = np.array([1.0, 0.0, 0.0], dtype=np.float32)
print(v.shape, v.ndim, v.dtype)
batch = np.vstack([v, v])
print(batch.shape)                    # (2, 3)
print(batch @ v)                      # matrix-vector, shape (2,)


## 1.22 OOP basics

```python
class Embedder:
    def __init__(self, model_name: str):
        self.model_name = model_name

    def embed_query(self, text: str) -> np.ndarray:
        ...
```

- `__init__` is not a constructor allocator; it initializes `self`.
- `self` is explicit (unlike C++ `this`).
- `@dataclass` generates `__init__` / `__repr__` / optional `__eq__`.
- Use a class when there is **lifecycle** (loaded model, growing index). Use a function when the operation is stateless (`cosine_similarity`).


In [ ]:
@dataclass(frozen=True)
class Chunk:
    id: str
    text: str
    source: str
    chunk_index: int
    document_id: str
    metadata: dict = field(default_factory=dict)

c = Chunk("d1_chunk_0", "hello", "a.txt", 0, "d1")
print(c)
# c.text = "x"   # FrozenInstanceError — good: chunks should not mutate after index


### Part 1 practice set (do these on paper)

1. Write `def word_counts(text: str) -> dict[str, int]` using `.get` (no `Counter`).
2. Write `def first_unique_char(s: str) -> int` (index or -1).
3. Given `chunks` and `embeddings` of possibly different lengths, raise a clear error (do not zip silently).
4. Sort retrieval results by score descending, `chunk_id` ascending.
5. Read all `*.txt` in a folder with pathlib, sorted, UTF-8.

### Assessment takeaway

If you can write `.get`, `set`, slices, `enumerate`, `zip(..., strict=True)`, `sorted(key=...)`, `Path.read_text`, `json.loads`, and a frozen dataclass without thinking, Part 1 is done.


# Part 2 — Python data manipulation for RAG

A RAG pipeline is a sequence of **records that must stay aligned**.

```
Document  {id, text, source}
    ↓ chunk
Chunk     {id, text, source, chunk_index, document_id}
    ↓ embed
Embedding {chunk_id, embedding}
    ↓ search
Result    {chunk, score}
```

If any join key is lost, retrieval returns the wrong passage with a confident score. That is the most common hidden-test failure.


In [ ]:
document = {"id": "doc1", "text": "Embeddings turn text into vectors.", "source": "file.txt"}

chunk = {
    "id": "doc1_chunk_0",
    "text": "Embeddings turn text into vectors.",
    "source": "file.txt",
    "chunk_index": 0,
    "document_id": document["id"],
}

embedding_row = {"chunk_id": chunk["id"], "embedding": [0.11, -0.02, 0.07]}

retrieval_result = {"chunk": chunk, "score": 0.87}

# Alignment invariant: every embedding_row.chunk_id exists in chunks
chunks_by_id = {chunk["id"]: chunk}
assert embedding_row["chunk_id"] in chunks_by_id


## Why metadata alignment matters

The user does not want a raw vector. They want **text + citation**. The score is useless if it is attached to the wrong chunk.

Keep one join key (`chunk.id`) and copy metadata **at chunk time**, not at retrieval time from a global that may have changed.

### Parallel arrays (works, easy to desync)

```python
texts = [...]
embeddings = [...]
ids = [...]
# texts[i] MUST correspond to embeddings[i]
```

### Parallel arrays + zip(strict=True) (acceptable)

### List of records (clearest)

```python
{"chunk": chunk, "embedding": vec}
```

### Two maps (good for lookup)

```python
chunk_by_id[id] = chunk
vector_by_id[id] = vec
```

Assessments often start with parallel arrays. **Validate lengths before you embed or search.**


In [ ]:
def align(chunks, embeddings):
    if len(chunks) != len(embeddings):
        raise ValueError(
            f"alignment error: {len(chunks)} chunks vs {len(embeddings)} embeddings"
        )
    return [
        {"chunk": c, "embedding": e}
        for c, e in zip(chunks, embeddings, strict=True)
    ]


# Common mistakes (do not ship these)

def lose_document_id(doc):
    # BAD: chunk id is only an integer
    return [{"id": 0, "text": doc["text"]}]


def mix_by_sort(chunks, embeddings):
    # BAD: sort texts but not vectors
    return sorted(chunks, key=lambda c: c["text"]), embeddings


def mutate_while_iterating(chunks):
    for c in chunks:
        if not c["text"].strip():
            chunks.remove(c)   # BAD
    return chunks


def inconsistent_metadata(doc):
    # BAD: source on the doc is file.txt, chunk says unknown
    return {"id": doc["id"] + "_chunk_0", "text": doc["text"], "source": "unknown"}


def duplicate_ids(docs):
    return [{"id": "doc", "text": d} for d in docs]  # every id is 'doc'


def missing_fields(text):
    return {"text": text}  # no id, no source


### Hidden tests in this area

- Two files with the same stem (`/a/notes.txt`, `/b/notes.txt`) → colliding `id="notes"`.
- Empty text after preprocess → chunk list empty, later `embeddings[0]` IndexError.
- `json.dumps` then `json.loads` turns tuples into lists; your embedding is still a list, fine; your tuple keys are gone.
- Reusing one `metadata={}` dict for every chunk (Part 1 identity bug).
- Sorting by similarity and returning `texts[best_i]` after you already filtered `texts` but not `embeddings`.

### Practice

Build `index = {chunk_id: (chunk, embedding)}` from two lists and reject duplicates.

### Assessment takeaway

Write the invariant in a comment and enforce it:

`assert len(chunks) == len(embeddings)` and `chunk.id` is unique.


# Part 3 — RAG-relevant DSA

You already know the algorithms. This section is **Python + why a RAG engineer uses it**.

For each problem: statement, intuition, brute force, optimal, implementation, complexities, mistakes, edges, hidden tests, RAG why, then runnable cases.

---

## Phase 1 — Python + hashing


### 3.1 Two Sum

**Statement.** `nums` and `target`. Return indices `i, j` (`i != j`) with `nums[i] + nums[j] == target`.

**Intuition.** For each value `x`, you want `target - x` already seen. That is a dict: value → index.

**Brute.** Double loop O(n²).

**Optimal.** One pass hash map O(n) time, O(n) space.

**RAG why.** "Find the stored vector that pairs with this query" is the same lookup shape: key → payload. Also: joining `chunk_id` to a row.


In [ ]:
def two_sum(nums, target):
    seen = {}
    for i, value in enumerate(nums):
        need = target - value
        if need in seen:
            return [seen[need], i]
        seen[value] = i
    return []


def _check_two_sum():
    cases = [
        ([2, 7, 11, 15], 9, [0, 1]),
        ([3, 2, 4], 6, [1, 2]),
        ([3, 3], 6, [0, 1]),          # duplicates
        ([], 0, []),
        ([1], 1, []),
        ([-1, -2, -3], -5, [1, 2]),   # negatives
        ([0, 4, 3, 0], 0, [0, 3]),
        ([10**9, 1, 10**9], 2 * 10**9, [0, 2]),
        ([1, 2, 3], 7, []),
    ]
    for nums, target, expected in cases:
        got = two_sum(nums, target)
        if expected == []:
            assert got == []
        else:
            i, j = got
            assert nums[i] + nums[j] == target
    print("two_sum ok")


_check_two_sum()


**Mistakes:** storing value→value instead of value→index; using the same index twice (`need == value` and `seen[value] == i`); `seen[value] = i` **before** the lookup when duplicates must pair.

**Hidden:** two zeros; `target` overflow if you think Python ints are 32-bit (they are not).

**Takeaway:** `enumerate` + dict lookup.


### 3.2 Contains Duplicate

**Statement.** True iff any value appears twice.

**Optimal.** `return len(nums) != len(set(nums))` or a growing set (early exit).

**RAG why.** Duplicate document IDs, duplicate chunk IDs, ingesting the same file twice.


In [ ]:
def contains_duplicate(nums):
    seen = set()
    for x in nums:
        if x in seen:
            return True
        seen.add(x)
    return False


assert contains_duplicate([1, 2, 3, 1]) is True
assert contains_duplicate([1, 2, 3, 4]) is False
assert contains_duplicate([]) is False
assert contains_duplicate([0]) is False
assert contains_duplicate([0, 0]) is True
assert contains_duplicate([-1, -1]) is True
assert contains_duplicate([10**12, 10**12 + 1]) is False
print("contains_duplicate ok")


**Mistake:** sorting in-place when the caller still needs original order. **Complexity:** O(n) / O(n). Sorting is O(n log n) and is worse unless you must not use extra memory.


### 3.3 Valid Anagram

**Statement.** Are `s` and `t` anagrams?

**Optimal.** `Counter(s) == Counter(t)` after length check. Or 26-slot array if only lowercase English.

**RAG why.** Token-set similarity, near-duplicate detection, "same bag of words after normalize".


In [ ]:
def valid_anagram(s, t):
    if len(s) != len(t):
        return False
    freq = {}
    for ch in s:
        freq[ch] = freq.get(ch, 0) + 1
    for ch in t:
        if ch not in freq:
            return False
        freq[ch] -= 1
        if freq[ch] == 0:
            del freq[ch]
    return not freq


assert valid_anagram("anagram", "nagaram") is True
assert valid_anagram("rat", "car") is False
assert valid_anagram("", "") is True
assert valid_anagram("a", "a") is True
assert valid_anagram("ab", "a") is False
assert valid_anagram("aacc", "ccac") is False
assert valid_anagram("Aa", "aA") is True   # case-sensitive here
assert valid_anagram("你好", "好你") is True
print("valid_anagram ok")


**Mistakes:** `sorted(s) == sorted(t)` is correct but O(n log n); lowercasing when the prompt is case-sensitive; using `set(s) == set(t)` (ignores counts).


### 3.4 Group Anagrams

**Statement.** Group strings that are anagrams.

**Optimal.** Key = `tuple(sorted(s))` or a 26-count tuple. `defaultdict(list)`.

**RAG why.** Near-duplicate chunks, grouping filenames, clustering after a cheap lexical key before you embed.


In [ ]:
def group_anagrams(strs):
    groups = defaultdict(list)
    for s in strs:
        groups[tuple(sorted(s))].append(s)
    return list(groups.values())


def _norm(groups):
    return sorted(sorted(g) for g in groups)


assert _norm(group_anagrams(["eat", "tea", "tan", "ate", "nat", "bat"])) == _norm(
    [["eat", "tea", "ate"], ["tan", "nat"], ["bat"]]
)
assert group_anagrams([""]) == [[""]]
assert group_anagrams(["a"]) == [["a"]]
assert _norm(group_anagrams(["ddddddddddg", "dgggggggggg"])) != _norm([["ddddddddddg", "dgggggggggg"]])
print("group_anagrams ok")


**Mistakes:** using `str(sorted(s))` (ugly but works); using `sorted(s)` as a dict key (`TypeError: unhashable`); mutating the input list.

**Hidden:** empty string group; unicode; 10k identical strings.


### 3.5 First Unique Character

**Statement.** Index of the first non-repeating character, else -1.

**Optimal.** Count, then second pass left-to-right.

**RAG why.** Deterministic tie-break, "first document that is not a duplicate", stream of IDs.


In [ ]:
def first_unique_char(s):
    counts = Counter(s)
    for i, ch in enumerate(s):
        if counts[ch] == 1:
            return i
    return -1


assert first_unique_char("leetcode") == 0
assert first_unique_char("loveleetcode") == 2
assert first_unique_char("aabb") == -1
assert first_unique_char("") == -1
assert first_unique_char("z") == 0
assert first_unique_char("cc") == -1
assert first_unique_char("aab") == 2
print("first_unique_char ok")


---

## Phase 2 — Text / window processing

These three problems are the mental model for **chunking and overlap**.


### 3.6 Longest Substring Without Repeating Characters

**Statement.** Length of the longest substring with all unique characters.

**Intuition.** Sliding window `[left, right]` + set/map of chars in the window. When a duplicate arrives, advance `left`.

**RAG why.** Deduping tokens inside a window; understanding why overlap windows move by `step = size - overlap`.


In [ ]:
def length_of_longest_substring(s):
    last = {}
    left = 0
    best = 0
    for right, ch in enumerate(s):
        if ch in last and last[ch] >= left:
            left = last[ch] + 1
        last[ch] = right
        best = max(best, right - left + 1)
    return best


assert length_of_longest_substring("abcabcbb") == 3
assert length_of_longest_substring("bbbbb") == 1
assert length_of_longest_substring("pwwkew") == 3
assert length_of_longest_substring("") == 0
assert length_of_longest_substring(" ") == 1
assert length_of_longest_substring("au") == 2
assert length_of_longest_substring("dvdf") == 3
assert length_of_longest_substring("abba") == 2
print("longest_substring ok")


**Mistakes:** forgetting to only move `left` forward (`abba`); using a set and `while` but forgetting to delete `s[left]`.

**Complexity:** O(n) time, O(min(n, alphabet)) space.


### 3.7 Minimum Window Substring

**Statement.** Smallest substring of `s` that covers every character in `t` (including duplicates).

**Intuition.** Expand right until the window is valid, then shrink left. Need a "how many unique constraints satisfied" counter.

**RAG why.** This is **coverage**: "smallest chunk that contains all query terms". Also the pattern for "grow until enough context, then shrink".


In [ ]:
def min_window(s, t):
    if not t or len(t) > len(s):
        return ""
    need = Counter(t)
    missing = len(need)
    left = 0
    best = (0, float("inf"))
    window = {}
    for right, ch in enumerate(s):
        window[ch] = window.get(ch, 0) + 1
        if ch in need and window[ch] == need[ch]:
            missing -= 1
        while missing == 0:
            if right - left < best[1] - best[0]:
                best = (left, right)
            left_ch = s[left]
            window[left_ch] -= 1
            if left_ch in need and window[left_ch] < need[left_ch]:
                missing += 1
            left += 1
    if best[1] is float("inf"):
        return ""
    i, j = best
    return s[i : j + 1]


assert min_window("ADOBECODEBANC", "ABC") == "BANC"
assert min_window("a", "a") == "a"
assert min_window("a", "aa") == ""
assert min_window("", "a") == ""
assert min_window("a", "") == ""
assert min_window("aa", "aa") == "aa"
assert min_window("bba", "ab") == "ba"
print("min_window ok")


**Mistakes:** treating `t` as a set (drops duplicate requirements); off-by-one on the stored bounds; shrinking before the window is valid.

**Hidden:** `t` longer than `s`; all characters the same; Unicode.


### 3.8 Merge Intervals

**Statement.** Merge overlapping `[start, end]` intervals.

**Intuition.** Sort by start. If `cur.start <= prev.end`, merge ends.

**RAG why.** Overlapping chunks, merging highlight ranges, collapsing adjacent passages before sending context to an LLM, detecting that `overlap >= chunk_size` never advances.


In [ ]:
def merge_intervals(intervals):
    if not intervals:
        return []
    intervals = sorted(intervals, key=lambda iv: iv[0])
    merged = [intervals[0][:]]
    for start, end in intervals[1:]:
        if start <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], end)
        else:
            merged.append([start, end])
    return merged


assert merge_intervals([[1, 3], [2, 6], [8, 10], [15, 18]]) == [[1, 6], [8, 10], [15, 18]]
assert merge_intervals([[1, 4], [4, 5]]) == [[1, 5]]      # touching
assert merge_intervals([]) == []
assert merge_intervals([[1, 1]]) == [[1, 1]]
assert merge_intervals([[1, 4], [2, 3]]) == [[1, 4]]      # contained
assert merge_intervals([[5, 6], [1, 2]]) == [[1, 2], [5, 6]]
assert merge_intervals([[1, 10], [2, 3], [4, 8]]) == [[1, 10]]
print("merge_intervals ok")


**Mistakes:** not sorting; `start < prev.end` instead of `<=` (depends on whether touching merges — usually yes); mutating the input without copying.

**Chunking connection.** A word window of size 4 overlap 1 is intervals `[0,4), [3,7), [6,10)`. Merge if you need a single span for citation.


---

## Phase 3 — Retrieval / ranking

You will implement top-k three ways:

1. `sorted(...)[:k]` — simple, O(n log n)
2. `heapq.nlargest` / `nsmallest` — O(n log k)
3. Full matrix multiply + `argpartition` — later, NumPy

`heapq` is a **min-heap**. For max-k by score, store `(-score, tie_key, item)` or use `nlargest`.


### 3.9 Top K Frequent Elements

**Statement.** Return the `k` most frequent numbers (order usually does not matter unless specified).

**Optimal.** Count O(n), then heap of size k, or bucket sort by frequency.

**RAG why.** Keyword retrieval, "most common terms in the corpus", cheap lexical baseline before embeddings.


In [ ]:
def top_k_frequent(nums, k):
    if k <= 0:
        return []
    counts = Counter(nums)
    return [val for val, _ in counts.most_common(k)]


assert sorted(top_k_frequent([1, 1, 1, 2, 2, 3], 2)) == [1, 2]
assert top_k_frequent([1], 1) == [1]
assert top_k_frequent([], 3) == []
assert top_k_frequent([1, 2], 0) == []
assert sorted(top_k_frequent([4, 4, 4, 5, 5, 6], 3)) == [4, 5, 6]  # k == n unique
assert top_k_frequent([1, 1, 2, 2], 1) in ([1], [2])               # ties
print("top_k_frequent ok")


**Mistakes:** `sorted(counts, key=counts.get)` without `reverse=True`; returning keys of the Counter in insertion order; not defining tie-break when the hidden test is strict.

**Hidden:** `k` larger than unique count — return all unique. `k = 0`. All frequencies equal.


### 3.10 Kth Largest Element

**Statement.** Return the k-th largest value in an unsorted array (1-based).

**Optimal.** Quickselect average O(n), or min-heap of size k O(n log k). Python: `heapq.nlargest(k, nums)[-1]`.

**RAG why.** Thresholding scores ("keep everything at least as good as the k-th"), selecting a cutoff.


In [ ]:
def kth_largest(nums, k):
    if not nums:
        raise ValueError("empty")
    if k < 1 or k > len(nums):
        raise ValueError("invalid k")
    return heapq.nlargest(k, nums)[-1]


assert kth_largest([3, 2, 1, 5, 6, 4], 2) == 5
assert kth_largest([3, 2, 3, 1, 2, 4, 5, 5, 6], 4) == 4
assert kth_largest([1], 1) == 1
assert kth_largest([-1, -2, -3], 1) == -1
assert kth_largest([-1, -2, -3], 3) == -3
assert kth_largest([7, 7, 7], 2) == 7
try:
    kth_largest([1, 2], 3)
    raise AssertionError
except ValueError:
    pass
print("kth_largest ok")


**Mistakes:** `sorted(nums)[k]` (that is k-th smallest, 0-based); mutating via `.sort()` when you must not; `k=0`.

**Complexity:** `nlargest` is O(n log k). Full sort is fine for n < ~10⁵ in an OA.


### 3.11 K Closest Points to Origin

**Statement.** `points[i] = [x, y]`. Return the k points with smallest Euclidean distance to `(0,0)`.

**Intuition.** This **is** nearest-neighbor search in 2-D. RAG is the same idea in 384-D with cosine instead of L2.

**Optimal.** Heap of size k, or `nlargest`/`nsmallest` on distance. Do not `sqrt` if you only compare (`x*x + y*y`).


In [ ]:
def k_closest(points, k):
    if k <= 0:
        return []
    k = min(k, len(points))
    # nsmallest keeps the k nearest; tie-break by coordinates for stability
    return heapq.nsmallest(
        k,
        points,
        key=lambda p: (p[0] * p[0] + p[1] * p[1], p[0], p[1]),
    )


assert k_closest([[1, 3], [-2, 2]], 1) == [[-2, 2]]
assert k_closest([[3, 3], [5, -1], [-2, 4]], 2) == [[3, 3], [-2, 4]]
assert k_closest([], 3) == []
assert k_closest([[1, 0]], 1) == [[1, 0]]
assert k_closest([[1, 0], [-1, 0]], 2) == [[-1, 0], [1, 0]]  # tie on distance
assert k_closest([[0, 0], [1, 1]], 5) == [[0, 0], [1, 1]]    # k > n
print("k_closest ok")


**RAG mapping**

```
point              → embedding
origin             → query vector (after you shift to a space where that makes sense)
Euclidean distance → 1 - cosine  (if vectors are L2-normalized, cosine rank == L2 rank)
k closest          → top_k
```

If embeddings are **L2-normalized**, `argmax cosine` == `argmin ||q - d||²`.


### 3.12 Find K Pairs with Smallest Sums

**Statement.** Two sorted arrays `nums1`, `nums2`. Return k pairs `(u, v)` with smallest `u+v`.

**Intuition.** The matrix of sums is sorted in both directions. Expand like Dijkstra / heap from `(0,0)`.

**RAG why.** Fusing two ranked lists (keyword score + dense score) without materializing the full cartesian product — the same "do not sort n² pairs" instinct as "do not sort the entire corpus when you only need top-k".


In [ ]:
def k_smallest_pairs(nums1, nums2, k):
    if not nums1 or not nums2 or k <= 0:
        return []
    heap = []
    visited = {(0, 0)}
    heapq.heappush(heap, (nums1[0] + nums2[0], 0, 0))
    out = []
    while heap and len(out) < k:
        _, i, j = heapq.heappop(heap)
        out.append([nums1[i], nums2[j]])
        if i + 1 < len(nums1) and (i + 1, j) not in visited:
            visited.add((i + 1, j))
            heapq.heappush(heap, (nums1[i + 1] + nums2[j], i + 1, j))
        if j + 1 < len(nums2) and (i, j + 1) not in visited:
            visited.add((i, j + 1))
            heapq.heappush(heap, (nums1[i] + nums2[j + 1], i, j + 1))
    return out


assert k_smallest_pairs([1, 7, 11], [2, 4, 6], 3) == [[1, 2], [1, 4], [1, 6]]
assert k_smallest_pairs([1, 1, 2], [1, 2, 3], 2) == [[1, 1], [1, 1]]
assert k_smallest_pairs([], [1], 3) == []
assert k_smallest_pairs([1], [], 3) == []
assert k_smallest_pairs([1], [1], 1) == [[1, 1]]
assert k_smallest_pairs([1, 2], [3], 3) == [[1, 3], [2, 3]]  # k > all pairs
assert k_smallest_pairs([1, 2], [1, 2], 0) == []
print("k_smallest_pairs ok")


**Mistakes:** generating all pairs then sorting (TLE when arrays are 10⁵); forgetting visited so you push `(i+1,j)` twice; assuming arrays are unsorted.

**Complexity:** O(k log k) with the heap expansion.

### Phase 3 takeaway

In the assessment you will write something equivalent to:

```python
heapq.nsmallest(k, range(n), key=lambda i: (-scores[i], ids[i]))
```

Memorize that line.


# Part 4 — NumPy for vector operations

You need enough NumPy to store embeddings and score them. Not broadcasting trivia.

| Idea | Meaning |
|------|---------|
| `np.array(xs)` | typed contiguous block |
| `.shape` | `(n, d)` batch of n vectors of dim d |
| `.ndim` | 1 for a query, 2 for a matrix |
| `.dtype` | prefer `float32` for storage, `float64` for scoring if you want fewer rounding surprises |
| `a + b`, `a * b` | elementwise |
| `np.dot(a, b)` / `a @ b` | inner product or matmul depending on ranks |
| `np.linalg.norm(a)` | L2 length |
| `axis=0` | down rows (per column) |
| `axis=1` | across columns (per row) |


In [ ]:
a = np.array([1.0, 2.0, 3.0])
print("shape", a.shape, "ndim", a.ndim, "dtype", a.dtype)
print("plus", a + 1, "times", a * 2)
print("dot", np.dot(a, a), "norm", np.linalg.norm(a))

batch = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
print("batch", batch.shape)
print("row norms", np.linalg.norm(batch, axis=1))
print("matmul batch @ [1,0]", batch @ np.array([1.0, 0.0]))

# normalize rows (safe)
norms = np.linalg.norm(batch, axis=1, keepdims=True)
safe = np.divide(batch, norms, out=np.zeros_like(batch), where=norms != 0)
print("normalized", safe)


## Cosine similarity

\[
\cos(\theta) = \frac{a \cdot b}{\|a\| \, \|b\|}
\]

| Pair | Score |
|------|-------|
| identical direction | `+1` |
| orthogonal | `0` |
| opposite | `-1` |
| any zero vector | **undefined** — we return `0.0` and document it |
| different dims | raise |
| NaN | raise |

If both vectors are L2-normalized, cosine **is** the dot product. FastEmbed's BGE models typically already L2-normalize. **Do not assume that** in an OA unless you check `np.linalg.norm(v) ≈ 1`.


In [ ]:
def cosine_similarity_python(a, b):
    if len(a) != len(b):
        raise ValueError(f"dimension mismatch: {len(a)} vs {len(b)}")
    if not a:
        raise ValueError("empty vector")
    dot = na = nb = 0.0
    for x, y in zip(a, b):
        dot += x * y
        na += x * x
        nb += y * y
    if na == 0.0 or nb == 0.0:
        return 0.0
    return dot / (math.sqrt(na) * math.sqrt(nb))


def cosine_similarity(a, b):
    va = np.asarray(a, dtype=np.float64).reshape(-1)
    vb = np.asarray(b, dtype=np.float64).reshape(-1)
    if va.size == 0 or vb.size == 0:
        raise ValueError("empty vector")
    if va.shape != vb.shape:
        raise ValueError(f"dimension mismatch: {va.shape} vs {vb.shape}")
    if not np.isfinite(va).all() or not np.isfinite(vb).all():
        raise ValueError("NaN/Inf")
    na = np.linalg.norm(va)
    nb = np.linalg.norm(vb)
    if na == 0.0 or nb == 0.0:
        return 0.0
    return float(np.dot(va, vb) / (na * nb))


assert abs(cosine_similarity([1, 0], [1, 0]) - 1) < 1e-9
assert abs(cosine_similarity([1, 0], [0, 1])) < 1e-9
assert abs(cosine_similarity([1, 0], [-1, 0]) + 1) < 1e-9
assert cosine_similarity([0, 0], [1, 0]) == 0.0
assert abs(cosine_similarity_python([1.0, 0.0], [1.0, 0.0]) - 1) < 1e-9
# floating point: almost parallel
print(cosine_similarity([1, 0], [1, 1e-12]))
try:
    cosine_similarity([1, 0], [1, 0, 0])
    raise AssertionError
except ValueError:
    pass
print("cosine ok")


**Why NumPy is preferred.** A Python loop over 10k vectors × 384 dims is ~4e6 multiplies in the interpreter. NumPy does it in C/BLAS. In an OA, a correct Python loop still passes if n is small; write NumPy if they imported it.

**Batch scoring**
```python
scores = (matrix @ query) / (np.linalg.norm(matrix, axis=1) * np.linalg.norm(query))
```
Guard zero norms.

**Takeaway:** raise on dim mismatch and NaN; define zero-vector behavior; vectorize the n×d loop.


# Part 5 — FastEmbed

Verified against current official docs: [FastEmbed README](https://github.com/qdrant/fastembed) and [Retrieval with FastEmbed](https://qdrant.github.io/fastembed/qdrant/Retrieval_with_FastEmbed/).

## What an embedding is

```
text  →  tokenizer + ONNX model  →  float vector in R^d  →  compare with cosine
```

The model maps meaning so that paraphrases land nearby. It is **not** a hash of characters (anagrams can be far; synonyms can be close).

- **Document / passage embedding:** vector for a chunk you will store.
- **Query embedding:** vector for the user's question.
- **Dimension `d`:** length of the vector. Default model `BAAI/bge-small-en-v1.5` has **d = 384**.
- **Same model + same method family** on both sides or scores are garbage.

## Current API (do not invent methods)

```python
from fastembed import TextEmbedding

model = TextEmbedding(model_name="BAAI/bge-small-en-v1.5")  # downloads once, then caches
model.embedding_size                    # 384
model.embed(texts, batch_size=256)      # generator of np.ndarray
model.passage_embed(texts)              # generator — use for documents
model.query_embed(query)                # generator — use for queries
TextEmbedding.list_supported_models()
```

Constructor extras you may see: `cache_dir`, `threads`, `providers=["CUDAExecutionProvider"]`, `lazy_load=True`.

**`embed()` returns a generator.** Nothing runs until you iterate. Always `list(...)` or `np.vstack(list(...))`.

**Retrieval tip from official docs:** prefer `query_embed` for queries and `passage_embed` for documents. BGE/E5-style models are trained with prefixes (`query:` / `passage:`). Using raw `embed()` on both sides is inconsistent for those families.

## CPU / batching

- Load the model **once**. Construction downloads + builds an ONNX session (seconds).
- `batch_size` trades RAM for throughput. 64–256 is typical on CPU.
- `parallel>1` or `parallel=0` is for offline encoding of large corpora, not for a single query.
- Stay on CPU unless the assessment says otherwise (`pip install fastembed`, not `fastembed-gpu`).


In [ ]:
# This cell talks to the real library if installed. It is safe to skip.
try:
    from fastembed import TextEmbedding
except ImportError:
    TextEmbedding = None
    print("fastembed not installed — skip live cells; read the API above")
else:
    print("fastembed import ok")
    print("default dim", TextEmbedding.get_embedding_size("BAAI/bge-small-en-v1.5"))


In [ ]:
FASTEMBED_READY = False
if TextEmbedding is not None:
    try:
        embed_model = TextEmbedding(model_name="BAAI/bge-small-en-v1.5")
        FASTEMBED_READY = True
        print("loaded", embed_model.model_name, "dim", embed_model.embedding_size)
    except Exception as exc:
        print("model load skipped:", type(exc).__name__, exc)


In [ ]:
if FASTEMBED_READY:
    one = list(embed_model.passage_embed(["Embeddings turn text into vectors."]))[0]
    print("one document", type(one), one.shape, one.dtype)

    many_texts = [
        "Embeddings turn text into vectors.",
        "Chunking splits long documents.",
    ]
    many = list(embed_model.passage_embed(many_texts, batch_size=2))
    matrix = np.vstack(many)
    print("batch", matrix.shape)

    query = next(iter(embed_model.query_embed("What is an embedding?")))
    print("query", np.asarray(query).shape)

    # generator pitfall
    gen = embed_model.embed(["once"])
    print("first materialize", len(list(gen)))
    print("second materialize", len(list(gen)))
else:
    print("skip live embed demos")


## Common FastEmbed mistakes

1. **`TextEmbedding()` inside a per-document loop** — minutes of reload, possible rate/disk hits.
2. **Query with `embed`, docs with `passage_embed`** (or two different `model_name`s).
3. **Forgetting `list()`** — you store a generator, later it is exhausted, `add_many` gets 0 rows.
4. **`np.array(list(model.embed(texts)))` vs `np.vstack`** — if the generator yields 1-D arrays, `np.array` gives `(n, d)` (good). If any yield is 2-D, shapes explode. Prefer `np.vstack`.
5. **Mixing models** in one index (384-d and 768-d).
6. **Huge `batch_size` on a 4GB laptop** — OOM. Shrink batch, not the idea.
7. **Storing Python lists of lists of Python floats** — works, wastes RAM. Store `float32` arrays.
8. **Re-embedding the same chunks** on every query. Embed corpus once; embed the query per request.
9. **Empty string batches** — some models still return a vector (the embedding of "empty"). Prefer skipping empty chunks before embed.
10. **Assuming `embed(["a", "b"])` returns a `list`** — it is an iterable. `len(model.embed(...))` may fail.

## Assessment takeaway

```python
class Embedder:
    def __init__(self):
        self.model = TextEmbedding(model_name="BAAI/bge-small-en-v1.5")
        self.dim = self.model.embedding_size
    def embed_documents(self, texts):
        return np.vstack(list(self.model.passage_embed(texts)))
    def embed_query(self, text):
        return np.asarray(next(iter(self.model.query_embed(text))))
```

Reuse one instance. Materialize. Check `shape[-1] == self.dim`.


# Part 6 — Ingestion + preprocessing

Ingestion is a **pure function of files → Document records**. No embeddings here.

```
path/dir → read bytes → decode → parse → Document(id, text, source, metadata)
```

Start with `.txt`. The same function signature should later accept `.json` and directories without rewriting callers.


In [ ]:
SUPPORTED = {".txt", ".json"}
_WS = re.compile(r"\s+")


def preprocess_text(text, *, lowercase=False, collapse_whitespace=True, strip=True):
    if not isinstance(text, str):
        raise TypeError("text must be a str")
    if strip:
        text = text.strip()
    if collapse_whitespace:
        text = _WS.sub(" ", text)
    if lowercase:
        text = text.lower()
    return text


def load_document(path, *, encoding="utf-8", preprocess=True):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if not path.is_file():
        raise IsADirectoryError(path)
    if path.suffix.lower() not in SUPPORTED:
        raise ValueError(f"unsupported extension {path.suffix}")

    raw = path.read_text(encoding=encoding)
    if path.suffix.lower() == ".txt":
        text = preprocess_text(raw) if preprocess else raw
        return {"id": path.stem, "text": text, "source": str(path), "metadata": {}}

    payload = json.loads(raw)
    if not isinstance(payload, dict) or not isinstance(payload.get("text"), str):
        raise ValueError("JSON must be an object with string field 'text'")
    text = preprocess_text(payload["text"]) if preprocess else payload["text"]
    return {
        "id": str(payload.get("id", path.stem)),
        "text": text,
        "source": str(payload.get("source", path)),
        "metadata": {k: v for k, v in payload.items() if k not in {"id", "text", "source"}},
    }


def load_documents(source, *, recursive=False):
    source = Path(source)
    if source.is_file():
        return [load_document(source)]
    if not source.is_dir():
        raise FileNotFoundError(source)
    files = source.rglob("*") if recursive else source.glob("*")
    files = sorted(p for p in files if p.is_file() and p.suffix.lower() in SUPPORTED)
    docs, seen = [], {}
    for p in files:
        doc = load_document(p)
        n = seen.get(doc["id"], 0)
        seen[doc["id"]] = n + 1
        if n:
            doc = {**doc, "id": f"{doc['id']}_{n}"}
        docs.append(doc)
    return docs


In [ ]:
from tempfile import TemporaryDirectory

with TemporaryDirectory() as td:
    root = Path(td)
    (root / "a.txt").write_text("Hello,\n\n  World!  ", encoding="utf-8")
    (root / "b.json").write_text(
        json.dumps({"id": "json1", "text": "JSON text", "topic": "demo"}),
        encoding="utf-8",
    )
    (root / "skip.md").write_text("ignored", encoding="utf-8")
    (root / "empty.txt").write_text("   \n", encoding="utf-8")
    docs = load_documents(root)
    print([(d["id"], d["text"], d["metadata"]) for d in docs])

# Should you lowercase / strip punctuation?
#   Search / bag-of-words homework: maybe.
#   Dense embeddings: usually NO. The model was trained on natural punctuation and casing.
#   "US" vs "us", "C++", emails, code identifiers all lose signal if you smash the text.


### Ingestion cases you must handle

| Case | Behavior |
|------|----------|
| UTF-8 text | default |
| empty file | `text=""` (do not crash) |
| repeated whitespace / newlines | collapse if preprocess on |
| Unicode | keep; decode UTF-8 |
| malformed JSON | `ValueError` |
| missing file | `FileNotFoundError` |
| `.pdf` | `ValueError` unsupported |
| duplicate stems | suffix `_1`, `_2` |
| directory order | `sorted` paths |
| huge file | OA: read all is OK; mention streaming as a follow-up |
| encoding errors | do not silently skip; fail or `errors="replace"` **explicitly** |

**Hidden:** `load_documents` on a file vs directory; Windows `\` vs `/` in `source` strings (use `Path`); BOM at start of UTF-8 file (`utf-8-sig`).

**Takeaway:** deterministic order, explicit encoding, metadata from the first moment, no aggressive normalize.


# Part 7 — Chunking

Chunking is its own component. Wrong overlap is a favorite hidden test.

```
document → chunk 0 → chunk 1 → chunk 2
```

Each chunk keeps `document_id`, `source`, `chunk_index`, `text`, and a **deterministic** `id = f"{doc_id}_chunk_{i}"`.


In [ ]:
def chunk_text(text, chunk_size=100, overlap=20, unit="word"):
    if chunk_size <= 0:
        raise ValueError("chunk_size must be > 0")
    if overlap < 0:
        raise ValueError("overlap must be >= 0")
    if overlap >= chunk_size:
        raise ValueError("overlap must be < chunk_size")
    if not text:
        return []
    tokens = text.split() if unit == "word" else list(text)
    joiner = " " if unit == "word" else ""
    if len(tokens) <= chunk_size:
        return [joiner.join(tokens)]
    step = chunk_size - overlap
    out, start, n = [], 0, len(tokens)
    while start < n:
        end = min(start + chunk_size, n)
        out.append(joiner.join(tokens[start:end]))
        if end == n:
            break
        start += step
    return out


def chunk_document(doc, chunk_size=100, overlap=20, unit="word"):
    texts = chunk_text(doc["text"], chunk_size, overlap, unit)
    return [
        {
            "id": f"{doc['id']}_chunk_{i}",
            "text": t,
            "source": doc["source"],
            "chunk_index": i,
            "document_id": doc["id"],
            "metadata": dict(doc.get("metadata") or {}),
        }
        for i, t in enumerate(texts)
    ]


print(chunk_text("a b c d e f", chunk_size=3, overlap=1))
print(chunk_text("short", chunk_size=10, overlap=2))
print(chunk_text("", chunk_size=10, overlap=2))
print(chunk_text("x" * 5, chunk_size=5, overlap=0, unit="char"))
try:
    chunk_text("hello world", chunk_size=3, overlap=3)
except ValueError as e:
    print(e)


### Beginner chunking mistakes

1. `step = overlap` instead of `step = chunk_size - overlap`.
2. Allowing `overlap == chunk_size` → infinite loop if you write `while start < n: start += 0`.
3. IDs like `hash(text)` — collide on duplicate paragraphs; not deterministic across processes if you use `id()`.
4. Dropping `source` so citations are impossible.
5. Character windows that split UTF-16 surrogates (Python 3 `str` is code points, usually OK; still do not slice bytes).
6. Re-joining words with `""` instead of `" "`.
7. Emitting an extra empty last chunk.
8. Using `range(0, n, chunk_size)` and forgetting overlap entirely.

| Case | Expected |
|------|----------|
| empty doc | `[]` |
| shorter than size | one chunk, original text (joined tokens) |
| exact size | one chunk |
| overlap 0 | adjacent, no shared tokens |
| overlap = size-1 | almost sliding-by-one |
| very large overlap (< size) | many chunks, still terminates |

**Takeaway:** validate sizes first, copy metadata, deterministic ids, test a 6-token toy by hand.


# Part 8 — Embedding pipeline

Lifecycle: construct once → embed documents in batches → embed each query with the **query** path.


In [ ]:
class Embedder:
    def __init__(self, model=None, dim=None):
        # In an OA you will typically do:
        # from fastembed import TextEmbedding
        # self.model = TextEmbedding(model_name="BAAI/bge-small-en-v1.5")
        # self.dim = self.model.embedding_size
        self.model = model
        self.dim = dim

    def embed_documents(self, texts, batch_size=256):
        texts = list(texts)
        if not texts:
            return np.empty((0, self.dim), dtype=np.float32)
        vectors = list(self.model.passage_embed(texts, batch_size=batch_size))
        matrix = np.vstack(vectors).astype(np.float32, copy=False)
        if matrix.shape[1] != self.dim:
            raise ValueError("unexpected embedding width")
        return matrix

    def embed_query(self, text):
        if not isinstance(text, str):
            raise TypeError("query must be str")
        vec = np.asarray(next(iter(self.model.query_embed(text))), dtype=np.float32)
        return vec.reshape(-1)


# Offline stand-in so later cells run without downloading a model
class FakeModel:
    def __init__(self, dim=8):
        self.dim = dim

    def _vec(self, text, salt):
        rng = np.random.default_rng(abs(hash(salt + text)) % (2**32))
        v = rng.normal(size=self.dim).astype(np.float32)
        n = np.linalg.norm(v)
        return v / n if n else v

    def passage_embed(self, texts, batch_size=256):
        for t in texts:
            yield self._vec(t, "passage:")

    def query_embed(self, query):
        yield self._vec(query, "query:")


fake = Embedder(model=FakeModel(8), dim=8)
print(fake.embed_documents([]).shape)
print(fake.embed_documents(["one chunk"]).shape)
print(fake.embed_documents(["a", "b", "a"]).shape)  # duplicates allowed, same text → same vec here
print(fake.embed_query("hello").shape)


**Tests to rehearse:** one chunk; many; empty list → `(0, d)`; empty string (decide: skip or embed); duplicates; large batch (does not crash); query type error.

**Memory:** `n * d * 4` bytes for float32. 100k × 384 ≈ 150 MB. Fine. 10M × 384 ≈ 15 GB. Not fine for a laptop OA.

**Takeaway:** empty-list shape, dim check, never construct the model per call.


# Part 9 — In-memory vector index (brute force)

No FAISS. Store parallel `chunks` + `embeddings`, search every row.

```
query → query embedding → score all rows → rank → top_k {chunk, score}
```

Complexity: **O(n · d)** for scoring + **O(n log k)** for the heap (or O(n log n) full sort).

Real vector DBs (HNSW, IVF, PQ) approximate this to sublinear n. In an OA, brute force is the correct first implementation. Mention the approximation as a follow-up sentence.


In [ ]:
class InMemoryVectorStore:
    def __init__(self):
        self.chunks = []
        self.embeddings = []
        self.ids = set()

    def add(self, chunk, embedding):
        vec = np.asarray(embedding, dtype=np.float64).reshape(-1)
        if vec.size == 0 or not np.isfinite(vec).all():
            raise ValueError(f"bad embedding for {chunk['id']}")
        if self.embeddings and vec.shape != self.embeddings[0].shape:
            raise ValueError("dimension mismatch")
        if chunk["id"] in self.ids:
            raise ValueError(f"duplicate id {chunk['id']}")
        self.chunks.append(chunk)
        self.embeddings.append(vec)
        self.ids.add(chunk["id"])

    def add_many(self, chunks, embeddings):
        embeddings = np.asarray(embeddings, dtype=np.float64)
        if len(chunks) != len(embeddings):
            raise ValueError("alignment error")
        for c, e in zip(chunks, embeddings, strict=True):
            self.add(c, e)

    def search(self, query_embedding, top_k=5, method="heap"):
        if top_k < 0:
            raise ValueError("top_k")
        if top_k == 0 or not self.chunks:
            return []
        matrix = np.vstack(self.embeddings)
        q = np.asarray(query_embedding, dtype=np.float64).reshape(-1)
        if q.shape != matrix.shape[1:]:
            # q is (d,), matrix (n, d)
            if q.size != matrix.shape[1]:
                raise ValueError("dimension mismatch")
        scores = []
        for i, row in enumerate(matrix):
            scores.append(cosine_similarity(q, row))
        k = min(top_k, len(self.chunks))
        key = lambda i: (-scores[i], self.chunks[i]["id"])
        if method == "sort":
            order = sorted(range(len(scores)), key=key)[:k]
        else:
            order = heapq.nsmallest(k, range(len(scores)), key=key)
        return [{"chunk": self.chunks[i], "score": float(scores[i])} for i in order]


store = InMemoryVectorStore()
store.add({"id": "c0", "text": "alpha", "source": "s", "chunk_index": 0, "document_id": "d"}, [1.0, 0.0])
store.add({"id": "c1", "text": "beta", "source": "s", "chunk_index": 1, "document_id": "d"}, [0.8, 0.2])
store.add({"id": "c2", "text": "gamma", "source": "s", "chunk_index": 2, "document_id": "d"}, [0.0, 1.0])
print(store.search([1.0, 0.0], top_k=2))
print(store.search([1.0, 0.0], top_k=9))   # k > n
print(store.search([1.0, 0.0], top_k=0))
print(InMemoryVectorStore().search([1.0, 0.0], top_k=3))


| Case | Behavior |
|------|----------|
| `top_k=1` | single best, ties broken by `chunk.id` |
| `top_k=n` | all rows, ranked |
| `top_k>n` | min(k, n) |
| `top_k=0` | `[]` |
| empty index | `[]` |
| duplicate vectors | both stored, both can return |
| duplicate ids | raise |
| zero vector | score 0 |
| dim mismatch | raise |
| NaN | raise on add |
| ties | deterministic id order |

**Takeaway:** validate k and dims; never return a vector without its chunk; O(n d) is the honest complexity.


# Part 10 — Retrieval and ranking

Two implementations you must be able to write from memory.


In [ ]:
def top_k_sort(scores, ids, k):
    if k <= 0:
        return []
    order = sorted(range(len(scores)), key=lambda i: (-scores[i], ids[i]))
    return order[: min(k, len(scores))]


def top_k_heap(scores, ids, k):
    if k <= 0:
        return []
    return heapq.nsmallest(min(k, len(scores)), range(len(scores)),
                           key=lambda i: (-scores[i], ids[i]))


scores = [0.2, 0.9, 0.9, 0.1]
ids = ["d", "b", "a", "c"]
print(top_k_sort(scores, ids, 2))   # indices of 0.9/'a' then 0.9/'b'
print(top_k_heap(scores, ids, 2))
print(top_k_sort([], [], 3))
print(top_k_sort(scores, ids, 0))

# When to use which
# n <= 5_000 or k ~ n  → sorted is clearer and fast enough
# n is huge and k is tiny (10) → heap
# OA with n=1000 chunks → sorted is what they expect unless they mention heapq


Connect back to DSA: this is **K Closest Points** with a custom key and **Top K Frequent**'s heap.

`sorted` is **stable** (equal keys keep original order). After you add `id` into the key, stability should not matter.

**Takeaway:** one `key=lambda i: (-score[i], id[i])` function shared by sort and heap.


# Part 11 — Module separation

Assessment signal: they want to see that `main.py` is 20 lines of wiring, not 400 lines of mixed concerns.

```
rag/
├── main.py
├── ingestion/loader.py
├── preprocessing/chunker.py
├── embeddings/embedder.py
├── retrieval/vector_store.py
├── models/document.py
└── utils/similarity.py
```

| Module | Responsibility | Knows about |
|--------|----------------|-------------|
| `models` | records only | nothing above |
| `utils.similarity` | math | NumPy |
| `ingestion` | files → Document | models, pathlib, json |
| `preprocessing` | Document → Chunk | models |
| `embeddings` | text → vectors | FastEmbed, models.Chunk |
| `retrieval` | store + search | models, similarity |
| `main` | wire the graph | all of the above |

**Dependency flow (one direction):**

```
main → ingestion → models
     → preprocessing → models
     → embeddings → models, fastembed
     → retrieval → models, utils
     → utils
```

`chunker` must not import `Embedder`. `vector_store` must not import `TextEmbedding`. `loader` must not import the store.

## Imports

```python
# good
from rag.preprocessing.chunker import chunk_documents
from rag.models.document import Document

# bad
from rag.main import MiniRAG   # lower layer importing app
from rag.embeddings.embedder import Embedder  # inside chunker.py
```

Circular import typical cause: `document.py` imports `vector_store` for a type hint and `vector_store` imports `Chunk`. Fix: keep types in `models/`.

## Classes vs functions

| Use a function | Use a class |
|----------------|-------------|
| `preprocess_text` | `Embedder` (heavy model) |
| `chunk_text` | `InMemoryVectorStore` (growing state) |
| `cosine_similarity` | `MiniRAG` (facade) |

## BAD architecture

```python
# monolith.py
model = TextEmbedding()
for path in glob("*.txt"):
    text = open(path).read().lower()
    for i in range(0, len(text), 50):
        v = list(TextEmbedding().embed([text[i:i+50]]))[0]  # reload + no overlap + no id
        db.append((path, v))
```

Problems: model in a loop, no metadata, no overlap, locale encoding, lowercase, list-of-tuples you cannot test.

## GOOD architecture

Each box has tests. `main` only calls `ingest` and `retrieve`.

The repo already contains this package under `rag/`. Read those files after you can rewrite them from scratch.


# Part 12 — Experienced RAG engineer mistakes

1. **Document / chunk / embedding misalignment** — You sorted chunks by length but not vectors. Retrieval cites the wrong paragraph. *Why:* silent `zip` of unequal lists. **Fix:** `strict=True` + length assert.

2. **Recreating the embedding model** — 2–10s × n files. *Why:* constructor is the expensive part. **Fix:** one `Embedder` in `main`.

3. **Different models for queries and documents** — dimensions may match by luck; geometry will not. *Why:* cosine in two incompatible spaces is meaningless.

4. **Wrong vector dimensions** — 384 vs 768. Crash if you check; silent garbage if you truncate.

5. **Incorrect normalization** — double-normalizing or normalizing only queries. Cosine is scale-invariant, so extra L2 usually OK; **inconsistent** extra scaling plus dot-product-as-score (forgetting to divide) is not.

6. **Zero-vector cosine** — `0/0 → NaN`. NaNs sort to the top or bottom arbitrarily. **Fix:** return 0 or skip the row; reject NaN on insert.

7. **Arbitrary tie results** — hidden tests expect a stable order. **Fix:** `(-score, chunk_id)`.

8. **Losing source during chunking** — you cannot cite. Automatic fail on "return metadata" rubrics.

9. **Duplicate chunk IDs** — later `add` overwrites or you have two rows with one key. **Fix:** reject duplicates.

10. **Incorrect overlap** — `start += overlap` or `start += chunk_size` only. Adjacent sentences vanish or infinite loop.

11. **`overlap >= chunk_size`** — window never advances. **Fix:** validate.

12. **Embedding empty chunks** — wasted slots that match oddly. **Fix:** drop empty text before embed.

13. **Swallowing file errors** — `except: continue` drops half the corpus. Tests seed a bad file on purpose.

14. **Assuming UTF-8 without saying so** — works on your machine. **Fix:** `encoding="utf-8"` always.

15. **Loading huge files entirely** — mention memory; for OA still read_text unless they say stream.

16. **Unnecessary normalization** — lowercasing legal/medical/code text.

17. **Changing meaning in preprocess** — deleting punctuation turns "don't" and "dont" into a policy decision you did not intend.

18. **Recomputing embeddings for identical text** — optional cache keyed by `(model_name, text, role)`. Do not cache across models.

19. **Sorting the entire index for top-k=3** — fine for n=500; say heap for n=1e6.

20. **Not validating `top_k`** — `top_k=-1` → `lst[:-1]` surprise, or `heapq` errors.

21. **Returning embeddings without metadata** — the caller prints floats.

22. **Mixing query/document preprocessing** — stemming the query but not docs (or the reverse).

23. **Model init inside a loop** — see (2). Same for `TextEmbedding()` in `embed_query`.

24. **Non-deterministic results** — `set` iteration order, unsorted `glob`, `dict` over ids you then take first. **Fix:** sort paths, explicit tie-break.

25. **Hidden module coupling** — chunker calls embedder "for convenience". You cannot unit-test chunking offline.

Each of these is a 30-second verbal follow-up. Prepare a one-line "why" and a one-line "fix".


# Part 13 — Testing / hidden cases

Write tests **before** you trust a chunker. Assessment often includes a hidden harness that looks like this.


In [ ]:
# pytest-style examples (pip install pytest to run as a suite)
try:
    import pytest
except ImportError:
    class pytest:  # demo stand-in
        class mark:
            @staticmethod
            def parametrize(*a, **k):
                return lambda fn: fn

        @staticmethod
        def raises(exc):
            class _C:
                def __enter__(self):
                    return None
                def __exit__(self, t, e, tb):
                    return t is not None and issubclass(t, exc)
            return _C()

        @staticmethod
        def approx(value):
            return value


def test_chunk_empty():
    assert chunk_text("", 100, 10) == []


def test_chunk_short():
    assert chunk_text("one two", 10, 2) == ["one two"]


@pytest.mark.parametrize(
    "text,size,overlap,expected",
    [
        ("a b c d e f", 3, 1, ["a b c", "c d e", "e f"]),
        ("a b c d", 2, 0, ["a b", "c d"]),
        ("hello", 2, 0, ["he", "ll", "o"]),  # if unit=char — see next
    ],
)
def test_chunk_windows(text, size, overlap, expected):
    if expected[-1] == "o":
        assert chunk_text(text, size, overlap, unit="char") == expected
    else:
        assert chunk_text(text, size, overlap) == expected


def test_invalid_overlap():
    with pytest.raises(ValueError):
        chunk_text("hello world", 4, 4)


def test_cosine_hidden():
    assert cosine_similarity([1, 0], [1, 0]) == pytest.approx(1)
    assert cosine_similarity([0, 0], [1, 1]) == 0.0
    with pytest.raises(ValueError):
        cosine_similarity([1], [1, 2])


def test_store_alignment():
    s = InMemoryVectorStore()
    with pytest.raises(ValueError):
        s.add_many(
            [{"id": "a", "text": "t", "source": "s", "chunk_index": 0, "document_id": "d"}],
            np.zeros((2, 3)),
        )

print("pytest-style examples defined")


Companion tests that already run: `tests/test_rag_components.py`.

```bash
python -m pytest tests/test_rag_components.py -q
```

### Per-component checklist

| Component | Happy | Boundary | Empty | Invalid | Duplicate | Large | Malformed |
|-----------|-------|----------|-------|---------|-----------|-------|-----------|
| loader | txt+json dir | 1 file | empty txt | missing, `.pdf` | same stem | 1MB txt | bad json, bad utf-8 |
| preprocess | spaces | already clean | `""` | `None` | — | huge ws | — |
| chunker | overlap 1 | exact size | `""` | overlap≥size | same text two docs | 10k tokens | bad unit |
| embedder | batch of 3 | 1 text | `[]` | non-str | same text twice | batch 1k | empty string policy |
| store | top_k=2 | k=n, k>n | empty store | k<0, dim | same id | n=5k | NaN vector |
| ranker | unique scores | ties | no scores | k=0 | duplicate scores | — | — |

**Takeaway:** if you cannot name a test for empty, k, and alignment, you are not assessment-ready.


# Part 14 — Build RAG from primitives

No LangChain. No LlamaIndex. This cell is a complete system you should be able to retype.

Architecture:

```
documents → loader → preprocessor → chunker → embedder → vector store → retriever → top-k
```

The same design lives on disk as the `rag/` package (`rag/main.py` is the facade). Below is a self-contained version for study.


In [ ]:
@dataclass(frozen=True)
class _Doc:
    id: str
    text: str
    source: str
    metadata: dict = field(default_factory=dict)


@dataclass(frozen=True)
class _Chunk:
    id: str
    text: str
    source: str
    chunk_index: int
    document_id: str


class MiniRAG:
    def __init__(self, embedder, chunk_size=40, overlap=8):
        self.embedder = embedder
        self.chunk_size = chunk_size
        self.overlap = overlap
        self.store = InMemoryVectorStore()

    def ingest_records(self, documents: list[dict]) -> int:
        chunks = []
        for doc in documents:
            for ch in chunk_document(doc, self.chunk_size, self.overlap):
                chunks.append(ch)
        if not chunks:
            return 0
        matrix = self.embedder.embed_documents([c["text"] for c in chunks])
        if len(chunks) != len(matrix):
            raise ValueError("embedder dropped or invented rows")
        self.store.add_many(chunks, matrix)
        return len(chunks)

    def retrieve(self, query: str, top_k: int = 5):
        q = self.embedder.embed_query(query)
        return self.store.search(q, top_k=top_k)


docs = load_documents(Path("data/sample_docs"))
print("loaded", [d["id"] for d in docs])

mini = MiniRAG(Embedder(model=FakeModel(16), dim=16), chunk_size=20, overlap=4)
n = mini.ingest_records(docs)
print("chunks", n)
for row in mini.retrieve("Why do we keep source metadata on chunks?", top_k=3):
    print(f"{row['score']:.3f} {row['chunk']['id']}: {row['chunk']['text'][:70]}")


### Swap FakeModel for FastEmbed

```python
from rag.embeddings.embedder import Embedder
from rag.main import MiniRAG

rag = MiniRAG(chunk_size=30, overlap=6)
rag.ingest("data/sample_docs")
rag.retrieve("What is cosine similarity used for?", top_k=3)
```

`python -m rag.main` runs that demo (requires `fastembed`).

### Robustness already in the package

- UTF-8 + JSON + directory load, sorted paths, colliding stems
- overlap validation
- alignment check in `add_many`
- cosine zero-vector and NaN guards
- heap top-k with id tie-break
- one `TextEmbedding` instance; `passage_embed` / `query_embed`

### What you must be able to write in 45 minutes

1. `preprocess_text` + `load_documents`
2. `chunk_text` with overlap
3. `cosine_similarity`
4. `InMemoryVectorStore.add/search`
5. A 15-line `MiniRAG` that calls them in order

If any of those five takes you more than 10 minutes, drill that file only.


# Part 15 — Assessment simulation

Work each problem **before** scrolling to its solution. Timebox 15–25 minutes each.

Shared conventions: UTF-8, fail fast, deterministic ids `{document_id}_chunk_{i}`, cosine on raw vectors, `top_k<=0` → `[]` except where noted.

---

## Q1 — Document loader

**Statement.** Implement `load_document(path: str | Path) -> dict`.

**Signature.** `def load_document(path: str | Path) -> dict:`

**Input.** Path to a `.txt` file.

**Output.** `{"id": <stem>, "text": <stripped collapsed-whitespace text>, "source": <str path>}`

**Constraints.** `encoding="utf-8"`. Raise `FileNotFoundError` / `ValueError` for missing or unsupported suffix.

**Hidden.** empty file; extra blank lines; `.TXT` suffix; path that is a directory.

**Traps.** forgetting stem vs name; not setting encoding; using `open` without `with`.


<details><summary>Q1 solution</summary>

```python
def load_document(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix.lower() != ".txt":
        raise ValueError("unsupported")
    text = re.sub(r"\s+", " ", path.read_text(encoding="utf-8")).strip()
    return {"id": path.stem, "text": text, "source": str(path)}
```
</details>


## Q2 — Clean and normalize documents

**Signature.** `def preprocess_text(text: str, lowercase: bool = False) -> str`

**Expected.** Strip, collapse any whitespace to a single space. Lowercase only if flagged.

**Hidden.** tabs + newlines; already-clean text; `""`; non-str → `TypeError`.

**Traps.** removing punctuation; `text.split()` then join which also collapses but drops intentional leading/trailing — usually OK; `lower` by default.


<details><summary>Q2 solution</summary>

```python
def preprocess_text(text, lowercase=False):
    if not isinstance(text, str):
        raise TypeError("text")
    text = re.sub(r"\s+", " ", text).strip()
    return text.lower() if lowercase else text
```
</details>


## Q3 — Chunk with overlap

**Signature.** `def chunk_text(text: str, chunk_size: int, overlap: int) -> list[str]`

**Input.** Whitespace-tokenized words. `chunk_size>0`, `0 <= overlap < chunk_size`.

**Expected.** Overlapping word windows. Last window may be shorter. Empty text → `[]`. Text shorter than size → `[text]` after a normal split-join (words separated by one space).

**Hidden.** overlap = size-1; exact multiple; one word; `overlap >= chunk_size` must raise.

**Traps.** infinite loop; `step = overlap`; character slicing when they asked words.


<details><summary>Q3 solution</summary>

```python
def chunk_text(text, chunk_size, overlap):
    if chunk_size <= 0 or overlap < 0 or overlap >= chunk_size:
        raise ValueError("bad sizes")
    words = text.split()
    if not words:
        return []
    if len(words) <= chunk_size:
        return [" ".join(words)]
    step, out, i, n = chunk_size - overlap, [], 0, len(words)
    while i < n:
        out.append(" ".join(words[i:i + chunk_size]))
        if i + chunk_size >= n:
            break
        i += step
    return out
```
</details>


## Q4 — Preserve metadata during chunking

**Signature.** `def chunk_document(doc: dict, chunk_size: int, overlap: int) -> list[dict]`

**Input.** `doc` has `id`, `text`, `source`, optional `metadata`.

**Expected.** Each chunk: `id`, `text`, `source`, `chunk_index`, `document_id`, `metadata` (copied).

**Hidden.** missing metadata key; shared metadata mutation (must copy); empty text → `[]`.

**Traps.** `metadata=doc["metadata"]` without copy; 1-based indexes.


<details><summary>Q4 solution</summary>

```python
def chunk_document(doc, chunk_size, overlap):
    meta = dict(doc.get("metadata") or {})
    return [
        {
            "id": f"{doc['id']}_chunk_{i}",
            "text": t,
            "source": doc["source"],
            "chunk_index": i,
            "document_id": doc["id"],
            "metadata": dict(meta),
        }
        for i, t in enumerate(chunk_text(doc["text"], chunk_size, overlap))
    ]
```
</details>


## Q5 — FastEmbed embeddings

**Signature.**
```python
class Embedder:
    def __init__(self, model_name: str = "BAAI/bge-small-en-v1.5"): ...
    def embed_documents(self, texts: list[str]) -> np.ndarray: ...
    def embed_query(self, text: str) -> np.ndarray: ...
```

**Expected.** One `TextEmbedding`. Documents via `passage_embed`, query via `query_embed`. Return `float32` arrays, shape `(n, d)` and `(d,)`. Empty document list → `(0, d)`.

**Hidden.** generator exhaustion if they cache the generator; dim check.

**Traps.** constructing the model in `embed_query`; using `embed` for one side only.


<details><summary>Q5 solution</summary>

```python
class Embedder:
    def __init__(self, model_name="BAAI/bge-small-en-v1.5"):
        from fastembed import TextEmbedding
        self.model = TextEmbedding(model_name=model_name)
        self.dim = self.model.embedding_size

    def embed_documents(self, texts):
        texts = list(texts)
        if not texts:
            return np.empty((0, self.dim), dtype=np.float32)
        return np.vstack(list(self.model.passage_embed(texts))).astype(np.float32)

    def embed_query(self, text):
        return np.asarray(next(iter(self.model.query_embed(text))), dtype=np.float32).reshape(-1)
```
</details>


## Q6 — In-memory vector store

**Signature.**
```python
class InMemoryVectorStore:
    def add(self, chunk: dict, embedding: np.ndarray) -> None: ...
    def add_many(self, chunks: list[dict], embeddings: np.ndarray) -> None: ...
    def search(self, query_embedding: np.ndarray, top_k: int) -> list[dict]: ...
```

**Output of search.** `[{"chunk": ..., "score": float}, ...]` length `min(top_k, n)`.

**Constraints.** Reject dim mismatch, duplicate ids, length mismatch, `top_k < 0`. Empty store or `top_k==0` → `[]`.

**Hidden.** k > n; ties; NaN embedding.

**Traps.** returning only scores; `zip` without length check.


<details><summary>Q6 solution</summary>

Use the `InMemoryVectorStore` from Part 9. Key lines: validate, `cosine_similarity` per row (or matrix), `heapq.nsmallest(k, range(n), key=lambda i: (-scores[i], chunks[i]["id"]))`.
</details>


## Q7 — Cosine similarity

**Signature.** `def cosine_similarity(a, b) -> float`

**Expected.** `+1` / `0` / `-1` on the textbook pairs. Zero vector → `0.0`. Dim mismatch or NaN → `ValueError`.

**Hidden.** lists vs arrays; row vectors shape `(1, d)`; very small floats.

**Traps.** integer division; forgetting to convert to float.


<details><summary>Q7 solution</summary>

See Part 4 `cosine_similarity`. Reshape to 1-D, check finite, norms, then `dot / (na*nb)`.
</details>


## Q8 — Retrieve top-k documents

**Signature.** `def retrieve(store, embedder, query: str, top_k: int) -> list[dict]`

**Expected.** Embed query once, `store.search(...)`.

**Hidden.** empty query string; `top_k` larger than corpus.

**Traps.** embedding every stored document again; reloading the model.


<details><summary>Q8 solution</summary>

```python
def retrieve(store, embedder, query, top_k):
    return store.search(embedder.embed_query(query), top_k=top_k)
```
</details>


## Q9 — Malformed documents

**Signature.** `def load_documents(directory: Path) -> list[dict]`

**Expected.** Load all `.txt` and well-formed `.json` objects with `text`. **Skip or raise?** Pick one and document it. Recommended for OAs: **raise** with the path in the message (fail fast). If they say "best effort", collect errors and continue.

**Hidden.** empty dir → `[]`; JSON array; JSON without `text`; unreadable bytes.

**Traps.** swallowing every exception; unsorted files.


<details><summary>Q9 solution</summary>

```python
def load_documents(directory):
    directory = Path(directory)
    files = sorted(p for p in directory.iterdir() if p.suffix.lower() in {".txt", ".json"})
    return [load_document(p) for p in files]  # load_document already validates
```
</details>


## Q10 — Refactor a monolith

**Statement.** Given a 80-line script that reads files, chunks, embeds, and searches in one function, split it into the Part 11 layout. `main` may only orchestrate.

**Hidden.** graders import `from rag.preprocessing.chunker import chunk_text` — that name must exist.

**Traps.** circular imports; putting FastEmbed inside the chunker "to be helpful".


<details><summary>Q10 solution</summary>

Copy the `rag/` package boundaries. `models` at the bottom. No downward imports.
</details>


## Q11 — Debug broken RAG code

Find the bugs (do not rewrite from scratch):

```python
model = None
def embed(text):
    global model
    model = TextEmbedding()
    return model.embed(text)          # generator, and reload every call

def search(q, docs, vecs):
    qv = embed(q)
    best = 0
    for i, v in zip(docs, vecs):      # zip not strict; qv is a generator
        if cosine(qv, v) > best:      # compares generator to array
            best = i
    return best                       # returns last doc object, not ranked list
```

**Traps.** fixing only one of: reload, generator, zip, top-1 instead of top-k, no metadata.


<details><summary>Q11 solution</summary>

1. Construct `TextEmbedding` once.
2. `np.asarray(next(iter(model.query_embed(q))))`.
3. `passage_embed` for `vecs`.
4. `zip(..., strict=True)` + length check.
5. Return `top_k` list of `{chunk, score}`.
</details>


## Q12 — Fix metadata / embedding alignment

```python
chunks.sort(key=lambda c: len(c["text"]))
index = list(zip(chunks, embeddings))
```

Embeddings were computed **before** the sort. Repair without recomputing if possible.

**Hidden.** duplicate lengths; empty chunks already removed from one list only.


<details><summary>Q12 solution</summary>

Never sort one side. If you must sort, zip first:

```python
paired = list(zip(chunks, embeddings, strict=True))
paired.sort(key=lambda p: len(p[0]["text"]))
```

Better: key by `chunk["id"]` on both sides.
</details>


## Q13 — Fix incorrect top-k

```python
scores = [cosine(q, v) for v in vecs]
return [chunks[i] for i in np.argsort(scores)[:k]]
```

**Traps.** `argsort` is ascending (worst first). No tie-break. `k > n` is OK here but `k=0` returns `[]` which is fine. Negative scores exist (cosine).


<details><summary>Q13 solution</summary>

```python
order = sorted(range(len(scores)), key=lambda i: (-scores[i], chunks[i]["id"]))
return [{"chunk": chunks[i], "score": scores[i]} for i in order[:min(k, len(scores))]]
```
</details>


## Q14 — Edge-case gauntlet

Implement `search(query_vec, matrix, chunks, top_k)` that survives:

- empty matrix
- `top_k` in `{0, 1, n, n+5, -1}` (`-1` raises)
- zero row in `matrix`
- dim mismatch
- two identical rows (deterministic id order)
- `chunks` / `matrix` length mismatch

No solution here on purpose — this **is** the drill. Your Part 9 store should already pass. If not, fix it.

---

## Exam-day checklist (memorize)

```
[ ] Path + encoding="utf-8" + sorted(glob)
[ ] preprocess = strip + collapse whitespace only
[ ] chunk: step = size - overlap; raise if overlap >= size
[ ] id = f"{doc_id}_chunk_{i}"; copy metadata
[ ] TextEmbedding once; passage_embed / query_embed; list() the generator
[ ] assert len(chunks) == len(embeddings)
[ ] cosine: dim check, zero → 0.0, no NaN
[ ] top_k: min(k,n); key = (-score, id); heap or sort
[ ] modules: models → utils → ingest/chunk/embed/store → main
```

You are ready when you can implement that list on a whiteboard in 20 minutes.


# Appendix — files in this repo

| Path | What |
|------|------|
| `rag_coding_prep.ipynb` | this notebook |
| `rag/` | Part 14 package |
| `data/sample_docs/` | tiny corpus |
| `tests/test_rag_components.py` | component tests (no FastEmbed required) |
| `rag_oa_practice.ipynb` | older drill notebook (stdlib TODOs) |

```bash
python -m pytest tests/test_rag_components.py -q
python -m rag.main          # needs fastembed
```
